# iMAT Cell-Specific Model Reconstruction

This notebook reconstructs cell-type-specific genome-scale metabolic models for astrocytes (iMMU1867a) and neurons (iMMU1867n) from the curated generic Mus musculus model (iMMU1867) using the iMAT algorithm.

iMAT (Integrative Metabolic Analysis Tool) uses cell-type proteomics data to identify which reactions are highly expressed, lowly expressed, or intermediately expressed in a given cell type, then builds a context-specific reconstruction that best matches that expression.

Proteomics input: Sharma et al. (2015) brain cell-type proteomics. These were preprocessed in steps 1-3 of this pipeline.

iMAT workflow followed: https://imatpy.readthedocs.io/en/latest/usage.html#workflow

Inputs:
- models/02_curated_iMMU1867/iMMU1867.xml
- results/04_cell_specific_iMAT/model_gene_to_proteomics_medians.xlsx (from steps 1-3)

Outputs:
- models/03_cell_specific_iMAT/iMMU1867a.xml (astrocyte)
- models/03_cell_specific_iMAT/iMMU1867n.xml (neuron)

In [1]:
# Imports

import pandas as pd
import numpy as np
from pathlib import Path
from imatpy.model_utils import read_model, write_model
from imatpy.parse_gpr import gene_to_rxn_weights
from imatpy.imat import imat
from imatpy.model_creation import generate_model

# All paths are relative to the repository root.
NOTEBOOK_DIR = Path.cwd()
MODEL_PATH = NOTEBOOK_DIR.parents[1] / "models" / "02_curated_iMMU1867" / "iMMU1867.xml"
MEDIANS_PATH = NOTEBOOK_DIR.parents[1] / "results" / "04_cell_specific_iMAT" / "model_gene_to_proteomics_medians.xlsx"

OUTPUT_ASTROCYTE = NOTEBOOK_DIR.parents[1] / "models" / "03_cell_specific_iMAT" / "iMMU1867a.xml"
OUTPUT_NEURON = NOTEBOOK_DIR.parents[1] / "models" / "03_cell_specific_iMAT" / "iMMU1867n.xml"

# iMAT parameters
EPSILON = 1e-3   # minimum flux for a reaction to be considered active
THRESHOLD = 1e-6   # reactions fluxes below this are considered off in the reconstructed model

In [2]:
# Load model and proteomics medians table

model = read_model(str(MODEL_PATH))
df = pd.read_excel(MEDIANS_PATH)

# Normalize column names (strip non-breaking spaces and whitespace)
df.columns = (
    df.columns
    .astype(str)
    .str.replace("\u00a0", " ", regex=False)
    .str.strip()
)

required = ["gene_name", "gene_id", "astrocyte_log2_median", "neuron_div15_log2_median"]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns in medians file: {missing}")

df["gene_id"] = df["gene_id"].astype(str).str.strip()

print(f"Model loaded: {len(model.reactions)} reactions, {len(model.metabolites)} metabolites, {len(model.genes)} genes")
print(f"\nMedians table: {len(df)} genes")

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2685367
Academic license 2685367 - for non-commercial use only - registered to sa___@wsu.edu
Model loaded: 10436 reactions, 5873 metabolites, 1867 genes

Medians table: 1867 genes


In [3]:
# Match proteomics gene IDs to model gene IDs
# Model genes may be prefixed with 'G_' (e.g. G_11461) while the proteomics table uses plain NCBI IDs.

model_gene_ids = [g.id for g in model.genes]
model_gene_set = set(model_gene_ids)

has_G_prefix = any(gid.startswith("G_") for gid in model_gene_ids)

def to_model_gene_id(ncbi_id: str) -> str:
    ncbi_id = str(ncbi_id).strip()
    if has_G_prefix:
        return f"G_{ncbi_id}"
    return ncbi_id

In [4]:
# Converts the proteomics expression into qualitative gene weights of -1, 0, or 1.
# Genes in the bottom 15% quantile get weight -1 (Low expression).
# Genes in the top 85% quantile get weight +1 (High expression).
# All other genes including genes with no proteomics data get weight 0 (Intermediate expression).

def build_gene_weights(df_in: pd.DataFrame, median_col: str, low_q=0.15, high_q=0.85) -> pd.Series:
    sub = df_in[["gene_id", median_col]].copy()

    detected = sub[median_col].dropna()
    if detected.empty:
        raise ValueError(f"No detected values in {median_col}")

    low_thr  = detected.quantile(low_q)   # bottom 15% threshold (low_q=0.15), genes below get -1
    high_thr = detected.quantile(high_q)  # top 15% starts at the 85th percentile (high_q=0.85), so genes at or above this get +1

    def assign_weight(x):
        if pd.isna(x):  return 0
        if x <= low_thr: return -1
        if x >= high_thr: return 1
        return 0

    sub["weight"] = sub[median_col].apply(assign_weight)
    sub["model_gene_id"] = sub["gene_id"].apply(to_model_gene_id)

    weights = pd.Series(0, index=model_gene_ids, dtype=float)
    mapped  = sub[sub["model_gene_id"].isin(model_gene_set)]
    weights.loc[mapped["model_gene_id"].values] = mapped["weight"].values

    return weights

In [5]:
# Build astrocyte gene weights, convert to reaction weights, and run iMAT

astro_gene_weights = build_gene_weights(df, "astrocyte_log2_median")
astro_rxn_weights  = gene_to_rxn_weights(model=model, gene_weights=astro_gene_weights)

astro_solution = imat(model=model, rxn_weights=astro_rxn_weights, epsilon=EPSILON, threshold=THRESHOLD)

print(f"Astrocyte iMAT objective: {astro_solution.objective_value}")
print(f"Astrocyte nonzero fluxes: {(astro_solution.fluxes.abs() > 0).sum()}")

Read LP format model from file C:\Users\SAMUEL~1.AYE\AppData\Local\Temp\tmp2ndxp05c.lp
Reading time = 0.06 seconds
: 5873 rows, 20872 columns, 79062 nonzeros
Astrocyte iMAT objective: 988.0000000271368
Astrocyte nonzero fluxes: 2134


In [6]:
# Generate and save the astrocyte context-specific reconstruction

astro_model = generate_model(
    model=model,
    rxn_weights=astro_rxn_weights,
    method="subset",
    epsilon=EPSILON,
    threshold=THRESHOLD
)

astro_model.id = "iMMU1867a"
astro_model.name = "Astrocyte-specific mouse GSMM"

OUTPUT_ASTROCYTE.parent.mkdir(parents=True, exist_ok=True)
write_model(astro_model, str(OUTPUT_ASTROCYTE))
print(f"Saved: {OUTPUT_ASTROCYTE}")
print(f"Astrocyte model: {len(astro_model.reactions)} reactions, {len(model.metabolites)} metabolites , {len(astro_model.genes)} genes")

Read LP format model from file C:\Users\SAMUEL~1.AYE\AppData\Local\Temp\tmp9q393zdr.lp
Reading time = 0.06 seconds
: 5873 rows, 20872 columns, 79062 nonzeros
Read LP format model from file C:\Users\SAMUEL~1.AYE\AppData\Local\Temp\tmpfhmo79e0.lp
Reading time = 0.06 seconds
: 5873 rows, 20872 columns, 79062 nonzeros
Saved: c:\Users\samuel.ayensu\Desktop\ecMMU1868-TPS\models\03_cell_specific_iMAT\iMMU1867a.xml
Astrocyte model: 10436 reactions, 5873 metabolites , 1867 genes


In [7]:
# Build neuron gene weights, convert to reaction weights, and run iMAT

neuron_gene_weights = build_gene_weights(df, "neuron_div15_log2_median")
neuron_rxn_weights  = gene_to_rxn_weights(model=model, gene_weights=neuron_gene_weights)

neuron_solution = imat(model=model, rxn_weights=neuron_rxn_weights, epsilon=EPSILON, threshold=THRESHOLD)

print(f"Neuron iMAT objective: {neuron_solution.objective_value}")
print(f"Neuron nonzero fluxes: {(neuron_solution.fluxes.abs() > 0).sum()}")

Read LP format model from file C:\Users\SAMUEL~1.AYE\AppData\Local\Temp\tmpd4wn9b3f.lp
Reading time = 0.09 seconds
: 5873 rows, 20872 columns, 79062 nonzeros
Neuron iMAT objective: 800.0
Neuron nonzero fluxes: 2039


In [8]:
# Generate and save the neuron context-specific reconstruction

neuron_model = generate_model(
    model=model,
    rxn_weights=neuron_rxn_weights,
    method="subset",
    epsilon=EPSILON,
    threshold=THRESHOLD
)

neuron_model.id = "iMMU1867n"
neuron_model.name = "Neuron-specific mouse GSMM"

OUTPUT_NEURON.parent.mkdir(parents=True, exist_ok=True)
write_model(neuron_model, str(OUTPUT_NEURON))
print(f"Saved: {OUTPUT_NEURON}")
print(f"Neuron model: {len(neuron_model.reactions)} reactions, {len(model.metabolites)} metabolites, {len(neuron_model.genes)} genes")

Read LP format model from file C:\Users\SAMUEL~1.AYE\AppData\Local\Temp\tmp5z2ika0h.lp
Reading time = 0.07 seconds
: 5873 rows, 20872 columns, 79062 nonzeros
Read LP format model from file C:\Users\SAMUEL~1.AYE\AppData\Local\Temp\tmpy8xh8_on.lp
Reading time = 0.09 seconds
: 5873 rows, 20872 columns, 79062 nonzeros
Saved: c:\Users\samuel.ayensu\Desktop\ecMMU1868-TPS\models\03_cell_specific_iMAT\iMMU1867n.xml
Neuron model: 10436 reactions, 5873 metabolites, 1867 genes


In [9]:
# Summary - gene and reaction weight counts for both cell types

def summarize_weights(gene_weights, rxn_weights, label):
    g = gene_weights.value_counts().to_dict()
    print(f"{label}")
    print(f"  Gene weights  - high (+1): {int(g.get(1.0, 0))}  neutral (0): {int(g.get(0.0, 0))}  low (-1): {int(g.get(-1.0, 0))}")
    print(f"  Reactions with nonzero weight: {int((rxn_weights != 0).sum())}")

summarize_weights(astro_gene_weights, astro_rxn_weights, "Astrocyte")
summarize_weights(neuron_gene_weights, neuron_rxn_weights, "Neuron")

Astrocyte
  Gene weights  - high (+1): 178  neutral (0): 1511  low (-1): 178
  Reactions with nonzero weight: 1065
Neuron
  Gene weights  - high (+1): 179  neutral (0): 1509  low (-1): 179
  Reactions with nonzero weight: 860
